# Gaussian Mixture Models (GMMs)

A flexible, probabilistic clustering and density estimation tool that represents data as coming from a combination of several Gaussian distributions. Unlike hard clustering methods, GMMs assign each point soft (probabilistic) membership across clusters, allowing for overlapping, elliptical cluster shapes.

<br>

<p align="center">
<img src="visualizations/GMM.png" width="600">
</p>

## Mathematical Foundation

**Goal:** Model the overall data density $p(\mathbf{x})$ as a weighted sum of $K$ Gaussians:

$$
p(\mathbf{x}) = \sum_{k=1}^K \pi_k \,\mathcal{N}(\mathbf{x}\mid\mu_k,\Sigma_k)
$$

where $\{\pi_k\}$ are mixing weights, $\{\mu_k\}$ are means, and $\{\Sigma_k\}$ are covariances.

**Soft Clustering:** Each data point $\mathbf{x}_n$ carries a "responsibility" $\gamma_{nk}\in[0,1]$ for each component $k$.

### The Generative Story

1. **Select a component** $k$ with probability $\pi_k$
2. **Draw a sample** $\mathbf{x}\sim\mathcal{N}(\mu_k,\Sigma_k)$

This models how the data could have been generated by first choosing one of $K$ "latent" processes, then sampling from that process's Gaussian.

### Why "Mixture"?

* **Multiple sources:** Real‑world data often arise from heterogeneous processes
* **Weighted sums:** We mix $K$ Gaussians so that the overall shape can approximate complex, multimodal densities

## Algorithm Steps: Expectation-Maximization (EM)

Because directly maximizing $\sum_n\ln\bigl[\sum_k\pi_k\,\mathcal{N}(\mathbf{x}_n)\bigr]$ is intractable, GMMs use EM:

1. **Initialize:** Choose initial parameters $\{\pi_k^{(0)}, \mu_k^{(0)}, \Sigma_k^{(0)}\}$

2. **E-step (Expectation):** Compute responsibilities:
   $$
   \gamma_{nk} = \frac{\pi_k\,\mathcal{N}(\mathbf{x}_n\mid\mu_k,\Sigma_k)}{\sum_j\pi_j\,\mathcal{N}(\mathbf{x}_n\mid\mu_j,\Sigma_j)}
   $$

3. **M-step (Maximization):** Update parameters:
   $$
   N_k = \sum_n \gamma_{nk}, \quad \pi_k \leftarrow \frac{N_k}{N}, \quad \mu_k \leftarrow \frac{1}{N_k}\sum_n \gamma_{nk}\,\mathbf{x}_n
   $$
   $$
   \Sigma_k \leftarrow \frac{1}{N_k}\sum_{n=1}^N \gamma_{nk}\,(\mathbf{x}_n - \mu_k)(\mathbf{x}_n - \mu_k)^\top
   $$

4. **Convergence:** Repeat E and M steps until log-likelihood improvement falls below threshold

## Advanced Topics

### Covariance Structures
* **Full:** Each cluster has arbitrary covariance $\Sigma_k$
* **Tied:** All clusters share one covariance
* **Diagonal:** $\Sigma_k$ diagonal (axis-aligned ellipsoids)
* **Spherical:** $\Sigma_k = \sigma_k^2 I$ (isotropic)

### Model Selection
* **BIC/AIC:** Penalized log-likelihood $\text{BIC} = -2\,\mathcal{L} + p\,\ln N$
* **Cross-validation:** Estimate held-out log-likelihood for different $K$

### Recovering K-Means

Under the special limit:
1. **Isotropic covariances:** $\Sigma_k = \sigma^2 I$
2. **Equal weights:** $\pi_k = 1/K$
3. **Variance → 0:** $\sigma^2\to0$

Then responsibilities $\gamma_{nk}$ become hard 0/1 assignments to the nearest mean, and the EM updates reduce exactly to Lloyd's K‑Means iterations.

## Key Characteristics

### Advantages
* Provides soft clustering (probabilistic membership)
* Can model elliptical and overlapping clusters
* Principled statistical foundation with EM algorithm
* Estimates cluster parameters and data density
* Flexible covariance structures

### Limitations
* Sensitive to initialization (may find local optima)
* Requires choosing number of components K
* Computationally expensive compared to K-means
* Assumes data follows Gaussian distributions
* Can struggle with high-dimensional data

### When to Use
* When clusters have different shapes/sizes
* For soft clustering and uncertainty quantification
* When you need density estimation
* For data with overlapping clusters
* When Gaussian assumption is reasonable

In [2]:
import numpy as np
from tqdm import tqdm
from scipy.stats import multivariate_normal
from coil20.coil20_utils import load_all_images, extract_images_tsne, flatten_images, extract_images_pca

import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)
from images.image_preprocessing import (
    extract_hog,
)

In [16]:
class GMM:
    def __init__(self, K, n_iter=100, tol=1e-4):
        self.K = K
        self.n_iter = n_iter
        self.tol = tol

    def initialize_parameters(self, X):
        N, D = X.shape
        self.N, self.D = N, D
        self.pi = np.ones(self.K) / self.K  # equal mixing weights
        indices = np.random.choice(N, self.K, replace=False)
        self.mu = X[indices]  # random initial means
        self.sigma = np.array([np.eye(D) for _ in range(self.K)])  # init covariances

    def e_step(self, X):
        # compute responsibilities
        self.gamma = np.zeros((self.N, self.K))
        for k in range(self.K):
            rv = multivariate_normal(self.mu[k], self.sigma[k])
            self.gamma[:, k] = self.pi[k] * rv.pdf(X)
        self.gamma /= self.gamma.sum(axis=1, keepdims=True)  # normalize

    def m_step(self, X):
        # update weights, means, covariances
        N_k = self.gamma.sum(axis=0)
        self.pi = N_k / self.N
        self.mu = (self.gamma.T @ X) / N_k[:, np.newaxis]
        for k in range(self.K):
            diff = X - self.mu[k]
            # weighted covariance
            self.sigma[k] = (self.gamma[:, k][:, np.newaxis] * diff).T @ diff / N_k[k]
            self.sigma[k] += np.eye(self.D) * 1e-6  # regularize

    def compute_log_likelihood(self, X):
        # total log-likelihood of the data
        ll = 0
        for k in range(self.K):
            rv = multivariate_normal(self.mu[k], self.sigma[k])
            ll += self.pi[k] * rv.pdf(X)
        return np.sum(np.log(ll))

    def fit(self, X):
        self.initialize_parameters(X)
        self.log_likelihoods = []
        for i in tqdm(range(self.n_iter), desc="GMM"):
            self.e_step(X)
            self.m_step(X)
            ll = self.compute_log_likelihood(X)
            self.log_likelihoods.append(ll)
            # stop if improvement is small
            if (
                i > 0
                and abs(self.log_likelihoods[-1] - self.log_likelihoods[-2]) < self.tol
            ):
                break

    def predict(self, X):
        # assign each point to the most probable component
        probs = np.zeros((X.shape[0], self.K))
        for k in range(self.K):
            rv = multivariate_normal(self.mu[k], self.sigma[k])
            probs[:, k] = self.pi[k] * rv.pdf(X)
        return np.argmax(probs, axis=1)

In [22]:
# Load data
X_images, y_labels = load_all_images()
X_pca = extract_images_tsne(extract_hog(X_images), n_components=3)

# Train algorithm
kmeans = GMM(K=20, n_iter=300)
kmeans.fit(X_pca)

# Evaluate
from sklearn.metrics import adjusted_rand_score

y_pred = kmeans.predict(X_pca)

ari = adjusted_rand_score(y_labels, y_pred)
print(f"Adjusted Rand Index: {ari:.4f}")

GMM:  18%|█▊        | 53/300 [00:00<00:01, 187.47it/s]

Adjusted Rand Index: 0.7031
